In [1]:
import pandas as pd
import re
import math

TOUR_NAME = "naic"

def remove_brackets(input_string):
    result = re.sub(r'\s*\[.*?\]\s*', '', input_string)
    return result

def normalize_name(name):
    name = name.lower()
    name = re.sub(r'[^a-z0-9_]', '_', name)
    return name

pairings_df = pd.read_csv('data.csv',sep='\t', header=None)
pairings_df.rename(columns={0:'Player',1:'Opponent',2:'Result',3:'Points',4:'Round'}, inplace=True)
pairings_df['Player'] = pairings_df['Player'].apply(remove_brackets).apply(normalize_name)
pairings_df['Opponent'] = pairings_df['Opponent'].apply(remove_brackets).apply(normalize_name)
pairings_df = pairings_df[(pairings_df['Opponent'] != 'BYE') & (pairings_df['Opponent'] != 'LATE')]

In [2]:
from bs4 import BeautifulSoup

def parse_limitless(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    table = soup.find('table', class_='data-table striped')
    data_tooltips = []
    for row in table.find_all('tr')[1:]:
        data_tooltips.append(row.find('span')['data-tooltip'])
    return data_tooltips

deck_df = pd.DataFrame()
for tournament in ['limitless.html']:
    with open(tournament, 'r', encoding='utf-8') as file:
        html_content = file.read()
        it_deck_df = pd.DataFrame()
        it_deck_df['Deck'] = parse_limitless(html_content)
        it_deck_df['Player'] = pairings_df['Player'].unique()[:len(it_deck_df)]
        placements = []
        for i in range(len(it_deck_df)):
            placements.append("Top {}".format(pow(2, math.ceil(math.log(i+1, 2))))) 
        it_deck_df['Placement'] = placements
    deck_df = pd.concat([deck_df, it_deck_df])

deck_dict = deck_df.set_index('Player')['Deck'].to_dict()


In [ ]:
matchups_df = pd.DataFrame(columns=['Deck', 'Opposing Deck', 'Wins', 'Losses', 'Ties'])
# pairings_final_df = pd.DataFrame(columns=['Player','Opponent','Result','Points','Round'])

for index, row in pairings_df.iterrows():
    try:
        player_deck = deck_dict[row['Player']]
        opp_deck = deck_dict[row['Opponent']]
        matchup = matchups_df.loc[(matchups_df['Deck'] == player_deck) & (matchups_df['Opposing Deck'] == opp_deck)]
        if row['Round'] == 9 and row['Points'] == 19 and row['Result'] == 'T':
            # print(row)
            continue
        if row['Result'] == 'T':
            if len(matchup) == 0:
                matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 1
            else:
                matchups_df.loc[matchup.index, 'Ties'] += 1
        elif row['Result'] == 'W':
            if len(matchup) == 0:
                matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 0, 0
            else:
                matchups_df.loc[matchup.index, 'Wins'] += 1
        elif row['Result'] == 'L':
            if len(matchup) == 0:
                    matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 1, 0
            else:
                matchups_df.loc[matchup.index, 'Losses'] += 1
        # pairings_final_df.loc[len(pairings_final_df)] = row
    except:
        continue


In [ ]:
with pd.ExcelWriter(f'datasets/{TOUR_NAME}.xlsx') as writer:
    # pairings_final_df.to_excel(writer, sheet_name='pairings', index=False)
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    matchups_df.to_excel(writer, sheet_name='matchups', index=False)